# Experiment 1 — YOLOv8-m Baseline

**Goal:** Establish per-class baseline mAP for marked vs non-marked speed bumps.

**Monitoring:** TensorBoard + Weights & Biases (wandb)

### Workflow:
1. Check GPU
2. Install libraries
3. Train on **original** data → monitor live
4. Train on **augmented** data → monitor live
5. Evaluate both per-class
6. Visualize predictions
7. Compare results & save

---
## 1. Check GPU

In [5]:
!nvidia-smi

'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


## 2. Install Libraries

In [ ]:
!pip install -q ultralytics
!pip install -q matplotlib pandas Pillow pyyaml
!pip install -q tensorboard
!pip install -q wandb

## 3. Login to Weights & Biases

If you don't have an account, create one free at [wandb.ai](https://wandb.ai).
You can also skip this cell and wandb will run in offline mode.

In [ ]:
import wandb

# This will ask for your API key the first time
# Get it from: https://wandb.ai/authorize
wandb.login()

## 4. Setup Paths & Constants

In [ ]:
import os
import torch

HOME = os.getcwd()

# Our two datasets
ORIGINAL_YAML = os.path.join(HOME, 'datasets', 'total-5', 'data.yaml')
AUGMENTED_YAML = os.path.join(HOME, 'datasets', 'total-5-augmented', 'data.yaml')

# Training settings
MAX_EPOCHS = 100
BATCH_SIZE = 4
IMG_SIZE = 640
PATIENCE = 20

# GPU check
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 5. Verify datasets exist

In [ ]:
import yaml

# Check original
with open(ORIGINAL_YAML, 'r') as f:
    orig_cfg = yaml.safe_load(f)
print('Original:', orig_cfg['names'])

# Check augmented
with open(AUGMENTED_YAML, 'r') as f:
    aug_cfg = yaml.safe_load(f)
print('Augmented:', aug_cfg['names'])

class_names = orig_cfg['names']
print('\nBoth datasets ready!')

## 6. Start TensorBoard

TensorBoard will show live training curves (loss, mAP, learning rate).
It updates automatically as training progresses.

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs/

---
# PART A — Train on Original Data (Baseline)

Imbalanced dataset. We expect high mAP for marked, low for non-marked.

## 7A. Load YOLOv8-m

In [ ]:
from ultralytics import YOLO

model_a = YOLO('yolov8m.pt')
print('YOLOv8-m loaded with COCO pretrained weights.')

## 8A. Train on original data

Watch TensorBoard above and W&B dashboard for live metrics.

In [ ]:
results_a = model_a.train(
    data=ORIGINAL_YAML,
    epochs=MAX_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project='runs',
    name='exp1a_original',
    patience=PATIENCE,
    save_period=10,
    plots=True,
    verbose=True
)

## 9A. Training curves

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt

results_img = os.path.join('runs', 'exp1a_original', 'results.png')

if os.path.exists(results_img):
    img = Image.open(results_img)
    plt.figure(figsize=(18, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Curves - Original Data', fontsize=16, fontweight='bold')
    plt.show()
else:
    print('Training plots not found yet.')

## 10A. Confusion matrix

In [ ]:
cm_path = os.path.join('runs', 'exp1a_original', 'confusion_matrix.png')

if os.path.exists(cm_path):
    img = Image.open(cm_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix - Original', fontsize=14, fontweight='bold')
    plt.show()
else:
    print('Confusion matrix not found.')

## 11A. Evaluate on test set (per-class mAP)

In [ ]:
# Load best weights
best_a = YOLO(os.path.join('runs', 'exp1a_original', 'weights', 'best.pt'))

# Test
metrics_a = best_a.val(
    data=ORIGINAL_YAML,
    split='test',
    device=DEVICE
)

In [ ]:
print('TEST RESULTS - ORIGINAL DATA')
print('=' * 40)
for i, name in enumerate(class_names):
    print(f'  {name}: AP50 = {metrics_a.box.ap50[i]:.4f}')
print(f'\n  Overall mAP50:    {metrics_a.box.map50:.4f}')
print(f'  Overall mAP50-95: {metrics_a.box.map:.4f}')

## 12A. Visualize predictions

In [ ]:
import random

test_img_dir = os.path.join(HOME, 'datasets', 'total-5', 'test', 'images')
test_images = [f for f in os.listdir(test_img_dir)
               if f.endswith('.jpg') or f.endswith('.png')]
chosen = random.sample(test_images, min(6, len(test_images)))

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, img_name in zip(axes, chosen):
    img_path = os.path.join(test_img_dir, img_name)
    result = best_a.predict(img_path, conf=0.5, verbose=False)[0]
    annotated = result.plot()[:, :, ::-1]  # BGR to RGB

    ax.imshow(annotated)
    ax.set_title(img_name[:25], fontsize=9)
    ax.axis('off')

plt.suptitle('Predictions - Original Data (Baseline)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 13A. Log results to W&B

In [ ]:
# Log final metrics to wandb for easy comparison
wandb.init(project='speed-bump-detection', name='exp1a_original', reinit=True)

wandb.log({
    'marked_AP50': metrics_a.box.ap50[0],
    'non_marked_AP50': metrics_a.box.ap50[1],
    'overall_mAP50': metrics_a.box.map50,
    'overall_mAP50_95': metrics_a.box.map,
    'experiment': 'original_baseline'
})

wandb.finish()
print('Results logged to W&B!')

---
# PART B — Train on Augmented Data

Same model, balanced dataset. We expect non-marked mAP to improve.

## 7B. Load fresh model

In [ ]:
model_b = YOLO('yolov8m.pt')
print('Fresh YOLOv8-m loaded for augmented experiment.')

## 8B. Train on augmented data

In [ ]:
results_b = model_b.train(
    data=AUGMENTED_YAML,
    epochs=MAX_EPOCHS,
    imgsz=IMG_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project='runs',
    name='exp1b_augmented',
    patience=PATIENCE,
    save_period=10,
    plots=True,
    verbose=True
)

## 9B. Training curves

In [ ]:
results_img = os.path.join('runs', 'exp1b_augmented', 'results.png')

if os.path.exists(results_img):
    img = Image.open(results_img)
    plt.figure(figsize=(18, 10))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Training Curves - Augmented Data', fontsize=16, fontweight='bold')
    plt.show()

## 10B. Confusion matrix

In [ ]:
cm_path = os.path.join('runs', 'exp1b_augmented', 'confusion_matrix.png')

if os.path.exists(cm_path):
    img = Image.open(cm_path)
    plt.figure(figsize=(8, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix - Augmented', fontsize=14, fontweight='bold')
    plt.show()

## 11B. Evaluate on test set

**Important:** We test on the **original** test set for fair comparison.

In [ ]:
best_b = YOLO(os.path.join('runs', 'exp1b_augmented', 'weights', 'best.pt'))

metrics_b = best_b.val(
    data=ORIGINAL_YAML,
    split='test',
    device=DEVICE
)

In [ ]:
print('TEST RESULTS - AUGMENTED DATA')
print('=' * 40)
for i, name in enumerate(class_names):
    print(f'  {name}: AP50 = {metrics_b.box.ap50[i]:.4f}')
print(f'\n  Overall mAP50:    {metrics_b.box.map50:.4f}')
print(f'  Overall mAP50-95: {metrics_b.box.map:.4f}')

## 12B. Visualize predictions

In [ ]:
# Same images from Part A for fair comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

for ax, img_name in zip(axes, chosen):
    img_path = os.path.join(test_img_dir, img_name)
    result = best_b.predict(img_path, conf=0.5, verbose=False)[0]
    annotated = result.plot()[:, :, ::-1]

    ax.imshow(annotated)
    ax.set_title(img_name[:25], fontsize=9)
    ax.axis('off')

plt.suptitle('Predictions - Augmented Data', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

## 13B. Log results to W&B

In [ ]:
wandb.init(project='speed-bump-detection', name='exp1b_augmented', reinit=True)

wandb.log({
    'marked_AP50': metrics_b.box.ap50[0],
    'non_marked_AP50': metrics_b.box.ap50[1],
    'overall_mAP50': metrics_b.box.map50,
    'overall_mAP50_95': metrics_b.box.map,
    'experiment': 'augmented'
})

wandb.finish()
print('Results logged to W&B!')

---
# PART C — Final Comparison

In [ ]:
import pandas as pd

comparison = pd.DataFrame({
    'Metric': [
        'marked AP50',
        'non-marked AP50',
        'Overall mAP50',
        'Overall mAP50-95'
    ],
    'Original': [
        round(metrics_a.box.ap50[0], 4),
        round(metrics_a.box.ap50[1], 4),
        round(metrics_a.box.map50, 4),
        round(metrics_a.box.map, 4)
    ],
    'Augmented': [
        round(metrics_b.box.ap50[0], 4),
        round(metrics_b.box.ap50[1], 4),
        round(metrics_b.box.map50, 4),
        round(metrics_b.box.map, 4)
    ]
})

comparison['Diff'] = comparison['Augmented'] - comparison['Original']

print('EXPERIMENT 1 — FINAL COMPARISON')
print('=' * 60)
print(comparison.to_string(index=False))

## Visual comparison

In [ ]:
import numpy as np

labels = ['marked\nAP50', 'non-marked\nAP50', 'overall\nmAP50']
orig_scores = [metrics_a.box.ap50[0], metrics_a.box.ap50[1], metrics_a.box.map50]
aug_scores = [metrics_b.box.ap50[0], metrics_b.box.ap50[1], metrics_b.box.map50]

x = np.arange(len(labels))
width = 0.35

fig, ax = plt.subplots(figsize=(10, 6))
bars1 = ax.bar(x - width/2, orig_scores, width, label='Original', color='#2196F3')
bars2 = ax.bar(x + width/2, aug_scores, width, label='Augmented', color='#4CAF50')

ax.set_ylabel('AP50')
ax.set_title('Original vs Augmented', fontsize=14, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()
ax.set_ylim(0, 1.0)

for bar in bars1 + bars2:
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
            f'{bar.get_height():.3f}', ha='center', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(HOME, 'exp1_comparison.png'), dpi=150)
plt.show()

## Save results

In [ ]:
comparison.to_csv(os.path.join(HOME, 'exp1_results.csv'), index=False)

print('Results saved to exp1_results.csv')
print('\nBest weights:')
print(f'  Original:  runs/exp1a_original/weights/best.pt')
print(f'  Augmented: runs/exp1b_augmented/weights/best.pt')
print(f'\nTensorBoard: Open the panel above to review all curves')
print(f'W&B Dashboard: https://wandb.ai (project: speed-bump-detection)')

## Decision for Experiment 2

Look at the **non-marked AP50** row above.

The best YOLOv8 non-marked AP50 is the number that **RT-DETR must beat** in Experiment 2.